# Exercise 8: South African Market Model

**Presenter** <br>
Priyesh Gosai <br>
Energy Systems Modeller <br>
priyesh@innovateimpact.com <br>

**Course Details**<br>
Modelling Integrated Power Markets <br>
Johannesburg, South Africa <br>
20-24 April 2026 <br>


In [ ]:
lesson_number = 8
print(f'lesson{lesson_number}')

## Google Colab Prep

In [ ]:
#@title Connect to Google Drive {display-mode:"form"}
CONNECT_TO_DRIVE = False #@param {type:"boolean"}

import os

if 'lesson_number' not in locals(): lesson_number = 8

if CONNECT_TO_DRIVE:
    from google.colab import drive
    # Mount Google Drive
    drive.mount('/content/drive')

    # Define the desired working directory path
    working_dir = f'/content/drive/MyDrive/ich-modelling-2026'
    lesson_folder = f'lesson-{lesson_number}'
    lesson_dir = os.path.join(working_dir, lesson_folder)

    # Create the working directory if it doesn't exist
    if not os.path.exists(working_dir):
        os.makedirs(working_dir)
        print(f"Directory '{working_dir}' created.")
    else:
        print(f"Directory '{working_dir}' already exists.")

    # Create the lesson directory if it doesn't exist
    if not os.path.exists(lesson_dir):
        os.makedirs(lesson_dir)
        print(f"Directory '{lesson_dir}' created.")
    else:
        print(f"Directory '{lesson_dir}' already exists.")

    # Change the current working directory to the lesson directory
    os.chdir(lesson_dir)

    print(f"Current working directory: {os.getcwd()}")
else:
    print("Not connecting to Google Drive.")

In [ ]:
#@title Install Packages {display-mode:"form"}
INSTALL_PACKAGES = False #@param {type:"boolean"}

import os

# Check if packages have already been installed in this session to prevent re-installation
if INSTALL_PACKAGES and not os.environ.get('PYPSA_PACKAGES_INSTALLED'):
  !pip install pypsa
  !pip install pypsa[excel] 
  !pip install folium mapclassify cartopy gurobipy
  os.environ['PYPSA_PACKAGES_INSTALLED'] = 'true'
elif not INSTALL_PACKAGES:
  print("Skipping package installation.")
else:
  print("PyPSA packages are already installed for this session.")

In [ ]:
#@title Download the file for this notebook {display-mode:"form"}
DOWNLOAD_FILE = False #@param {type:"boolean"}

import urllib.request
import os

if DOWNLOAD_FILE:
    url ="https://raw.github.com/PriyeshGosai/ich_course_2026/f756a8b64ce30fde1118e8bdd48f4388c78683b1/n_03_za_network_v3.xlsx"
    file_name = url.split('/')[-1]  # Extract filename from URL
    filename = file_name
    urllib.request.urlretrieve(url, filename)
    print(f"File downloaded successfully: {os.path.abspath(filename)}")

else:
    print("Skipping file download.")

## Functions

In [ ]:
def setup_gurobi_license_file():
    from google.colab import userdata
    import os
    
    wls_access_id = userdata.get('WLSACCESSID')
    wls_secret = userdata.get('WLSSECRET')
    license_id = userdata.get('LICENSEID')
    
    license_content = f"""WLSACCESSID={wls_access_id}
                        WLSSECRET={wls_secret}
                        LICENSEID={license_id}
                        """
    
    os.makedirs('/opt/gurobi', exist_ok=True)
    with open('/opt/gurobi/gurobi.lic', 'w') as f:
        f.write(license_content)
    
    print("✓ License file created")
    print(f"  License ID: {license_id}")


## Model

In [ ]:
start_timestamp = '2026-04-01 00:00:00'
n_days = 7

solver_name = 'gurobi'  # 'gurobi' or 'highs'
setup_gurobi_license_file()
unit_commitment = True # True or False for unit commitment of dispatchable generators 



In [ ]:
import gurobipy as gp
import pypsa
import pandas as pd
import linopy
import plotly.graph_objects as go

import linopy.solvers
linopy.solvers.gurobipy = gp
pypsa.options.api.new_components_api = True

pd.set_option('plotting.backend', 'plotly')

start = pd.Timestamp(start_timestamp)
end = start + pd.Timedelta(days=n_days) - pd.Timedelta(hours=1)

solver_options={
        'TimeLimit': 600,        # 10 minutes
        'MIPGap': 0.02,          # Or accept 2% gap (0.02)
        'LogToConsole': 1 
        }


n = pypsa.Network(file_name)

n.sanitize()

n.set_snapshots(pd.date_range(start, end, freq='h'))

dispatchable_carriers = ['coal','ocgt_diesel','ocgt_avf','sasol_gas'] 
n.generators.static.loc[n.generators.static.carrier.isin(dispatchable_carriers), 
                        'committable'] = unit_commitment

# Cluster the network spatially by bus location, and add load shedding to the clustered network
busmap = n.buses.static['location'].to_dict()  
n_clustered = n.cluster.spatial.cluster_by_busmap(busmap=busmap)

all_busmap = {bus: bus for bus in n.buses.static.index}
n_unconstrained = n.cluster.spatial.cluster_by_busmap(busmap=all_busmap)

n_clustered.optimize(solver_name=solver_name, include_objective_constant = False)
n_unconstrained.optimize(solver_name='highs', include_objective_constant = False)


In [ ]:
# Extract marginal prices for clustered network
supply_gens_clustered = n_clustered.generators.static[n_clustered.generators.static.type == 'Supply'].index.tolist()
supply_storage_clustered = n_clustered.storage_units.static.index.tolist()

supplier_volume_clustered = pd.concat([
    n_clustered.generators.dynamic.p[supply_gens_clustered],
    n_clustered.storage_units.dynamic.p[supply_storage_clustered].clip(lower=0)
], axis=1)

# Build supplier_prices_clustered efficiently
price_list_clustered = []
for gen in supply_gens_clustered:
    if gen in n_clustered.generators.dynamic.marginal_cost.columns:
        price_list_clustered.append(n_clustered.generators.dynamic.marginal_cost[[gen]])
    else:
        price_list_clustered.append(pd.Series(n_clustered.generators.static.loc[gen, 'marginal_cost'], 
                                               index=n_clustered.snapshots, name=gen).to_frame())

for storage in supply_storage_clustered:
    if storage in n_clustered.storage_units.dynamic.marginal_cost.columns:
        price_list_clustered.append(n_clustered.storage_units.dynamic.marginal_cost[[storage]])
    else:
        price_list_clustered.append(pd.Series(n_clustered.storage_units.static.loc[storage, 'marginal_cost'], 
                                               index=n_clustered.snapshots, name=storage).to_frame())

supplier_prices_clustered = pd.concat(price_list_clustered, axis=1)
clustered_marginal_prices = supplier_prices_clustered[supplier_volume_clustered > 0].max(axis=1)

# Extract marginal prices for unconstrained network
supply_gens_unconstrained = n_unconstrained.generators.static[n_unconstrained.generators.static.type == 'Supply'].index.tolist()
supply_storage_unconstrained = n_unconstrained.storage_units.static.index.tolist()

supplier_volume_unconstrained = pd.concat([
    n_unconstrained.generators.dynamic.p[supply_gens_unconstrained],
    n_unconstrained.storage_units.dynamic.p[supply_storage_unconstrained].clip(lower=0)
], axis=1)

# Build supplier_prices_unconstrained efficiently
price_list_unconstrained = []
for gen in supply_gens_unconstrained:
    if gen in n_unconstrained.generators.dynamic.marginal_cost.columns:
        price_list_unconstrained.append(n_unconstrained.generators.dynamic.marginal_cost[[gen]])
    else:
        price_list_unconstrained.append(pd.Series(n_unconstrained.generators.static.loc[gen, 'marginal_cost'], 
                                                   index=n_unconstrained.snapshots, name=gen).to_frame())

for storage in supply_storage_unconstrained:
    if storage in n_unconstrained.storage_units.dynamic.marginal_cost.columns:
        price_list_unconstrained.append(n_unconstrained.storage_units.dynamic.marginal_cost[[storage]])
    else:
        price_list_unconstrained.append(pd.Series(n_unconstrained.storage_units.static.loc[storage, 'marginal_cost'], 
                                                   index=n_unconstrained.snapshots, name=storage).to_frame())

supplier_prices_unconstrained = pd.concat(price_list_unconstrained, axis=1)
unconstrained_marginal_prices = supplier_prices_unconstrained[supplier_volume_unconstrained > 0].max(axis=1)

In [ ]:
# Create plotly figure
fig = go.Figure()

# Add clustered network trace
fig.add_trace(go.Scatter(
    x=n_clustered.snapshots,
    y=clustered_marginal_prices,
    mode='lines',
    name='Clustered Network',
    line=dict(color='blue', width=2)
))

# Add unconstrained network trace
fig.add_trace(go.Scatter(
    x=n_unconstrained.snapshots,
    y=unconstrained_marginal_prices,
    mode='lines',
    name='Unconstrained Network',
    line=dict(color='red', width=2)
))

# Update layout
fig.update_layout(
    title='Marginal Prices: Clustered vs Unconstrained Network',
    xaxis_title='Time',
    yaxis_title='Marginal Price (ZAR/MWh)',
    hovermode='x unified',
    template='plotly_white',
    height=600,
    width=1200
)

fig.show()